# Semi-supervised learning: few labels

```{admonition} Question
:class: tip
With 30 labelled hexagons, do the other 1,429 help?
```

**In this notebook:** 1 · choose 30 hexagons  2 · self-training  3 · one run  4 · twelve draws  5 · a cautious model  6 · label propagation

## 1 · Choose 30 hexagons

A label costs a field visit, and the budget covers 30. Two ways to choose them: at random, or 10 per class, which needs an older map of the classes.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

import helpers as h

hexes, X, y, W = h.load()        # the table, the features, the labels, the neighbours
SEED = 0
N_LABELS = 30       # try 15, 60, 150
THRESHOLD = 0.80    # how sure the model must be before its own prediction counts as a label

In [ ]:
samples = {"30 at random": h.draw_labelled(y, N_LABELS, SEED, per_class=False),
           "10 per class": h.draw_labelled(y, N_LABELS, SEED, per_class=True)}
print("true shares of the classes (%):", (pd.Series(y).value_counts(normalize=True).reindex(h.CLASSES) * 100).round(0).astype(int).to_dict())
h.map_two_samples(hexes, y, samples);

**Result.** Ten per class makes a third of the labelled hexagons `work`. On the map `work` is a tenth. That sample is skewed.

## 2 · Self-training

**Self-training** is a loop: fit a model on the labelled hexagons, predict the others, keep the predictions with a probability of at least 80% as **pseudo-labels**, repeat. The model is logistic regression on a hexagon's own counts and its neighbours' counts (`W @ X`, the spatial lag of week 3).

In [ ]:
Z = StandardScaler().fit_transform(np.hstack([X, W @ X]))     # 32 numbers per hexagon. Scaling needs no labels.


def self_train(F, y_start, threshold=0.80, C=1.0, max_rounds=40):
    """y_start holds a class name where the label is known and None where it is hidden.
    Returns the final model and a list with the label array after every round."""
    y_now = y_start.copy()
    history = [y_now.copy()]
    for _ in range(max_rounds):
        known = pd.notna(y_now)
        if known.all():
            break
        model = h.new_model(C).fit(F[known], y_now[known].astype(str))   # 1. fit on what we have
        proba = model.predict_proba(F[~known])                           # 2. predict the rest
        sure = proba.max(axis=1) >= threshold                            # 3. keep the confident predictions
        if not sure.any():
            break
        y_now[np.where(~known)[0][sure]] = model.classes_[proba.argmax(axis=1)[sure]]
        history.append(y_now.copy())                                     # 4. repeat
    known = pd.notna(y_now)
    return h.new_model(C).fit(F[known], y_now[known].astype(str)), history

## 3 · One run, round by round

The 30 hexagons drawn at random. **Accuracy** is the share of right answers on the hexagons whose label was hidden.

In [ ]:
labelled = samples["30 at random"]
y_start = h.hide_all_but(y, labelled)                 # the label where it is known, None elsewhere
hidden = pd.isna(y_start)
supervised = h.new_model().fit(Z[labelled], y[labelled])
model_self, history = self_train(Z, y_start, THRESHOLD)
for r, y_r in enumerate(history[1:], 1):
    adopted = pd.notna(y_r) & hidden
    print(f"round {r}: {pd.notna(y_r).sum():5d} hexagons carry a label, {(y_r[adopted] == y[adopted]).mean():.0%} of the pseudo-labels are right")
for name, model in (("supervised", supervised), ("self-training", model_self)):
    print(f"{name}: accuracy {(model.predict(Z)[hidden] == y[hidden]).mean():.1%}")
h.map_self_training(hexes, y, y_start, supervised.predict(Z), model_self.predict(Z));

<div class="frames" data-name="selftraining" data-prefix="round" data-first="0"></div>

*Grey: no label yet. Outlined: the 30 we started with.*

**Result.** After one round 1,238 hexagons carry a label, and about one pseudo-label in four is wrong. The wrong ones train the next round. Accuracy: 72.6% before, 72.7% after.

## 4 · Twelve draws

One draw can be luck. Twelve sets of 30 hexagons for each way of choosing them, and the share of hexagons that each model calls `work`.

In [ ]:
def one_draw(seed, per_class, C=1.0):
    lab = h.draw_labelled(y, N_LABELS, seed, per_class)
    start = h.hide_all_but(y, lab)
    hid = pd.isna(start)
    sup = h.new_model(C).fit(Z[lab], y[lab]).predict(Z[hid])
    st = self_train(Z, start, THRESHOLD, C=C)[0].predict(Z[hid])
    return {"supervised": (sup == y[hid]).mean(), "self-training": (st == y[hid]).mean(),
            "work, supervised": (sup == "work").mean(), "work, self-training": (st == "work").mean()}


draws = pd.DataFrame([{"sample": name, "seed": s, **one_draw(s, per_class)}
                      for name, per_class in (("30 at random", False), ("10 per class", True)) for s in range(12)])
means = draws.groupby("sample", sort=False).mean(numeric_only=True).drop(columns="seed")
(means * 100).round(1)

**Result.** At random: 73.5% and 73.8%, no gain. Ten per class: 70% becomes 74%.

In [ ]:
h.plot_draws(draws, (y == "work").mean());

The right panel gives the reason. The supervised model learns the skewed proportions and answers `work` for 15% of the hexagons. The unlabelled hexagons pull that back towards the 10% of the map. Self-training repaired a skewed sample. It found no new knowledge.

## 5 · A more cautious model

Logistic regression has a setting `C` for how far it trusts its training data. With 30 labels and 32 features a smaller `C` is a sensible choice.

In [ ]:
cautious = pd.DataFrame([{"sample": name, **one_draw(s, per_class, C=0.1)}
                         for name, per_class in (("30 at random", False), ("10 per class", True)) for s in range(12)])
(cautious.groupby("sample", sort=False).mean(numeric_only=True) * 100).round(1)

**Result.** With ten per class the cautious model reaches 74% without any unlabelled hexagon. Self-training brings it down to 45%: each round trains on the mistakes of the round before. Test the method on your own data before you trust it.

## 6 · Label propagation

Geography alone: ignore the counts and let every labelled hexagon pass its label to its neighbours through `W`, round after round. This is **label propagation**.

In [ ]:
def propagate(y_start, W, rounds=300, keep=0.9):
    start = np.stack([(y_start == c).astype(float) for c in h.CLASSES], axis=1)   # one column per class
    score = start.copy()
    for _ in range(rounds):
        score = keep * (W @ score) + (1 - keep) * start      # take the neighbours' average, keep the known labels
    pred = np.array(h.CLASSES, dtype=object)[score.argmax(axis=1)]
    pred[score.sum(axis=1) == 0] = None                      # hexagons no label ever reached
    return pred


pred_geo = propagate(y_start, W)
h.map_propagation(hexes, y, y_start, pred_geo);

In [ ]:
acc_geo = (pred_geo[hidden] == y[hidden]).mean()
acc_geo_draws = [np.mean(propagate(h.hide_all_but(y, lab), W)[np.setdiff1d(np.arange(len(y)), lab)] == np.delete(y, lab))
                 for lab in (h.draw_labelled(y, N_LABELS, s) for s in range(12))]
print(f"label propagation: accuracy {acc_geo:.1%} in this draw, {np.mean(acc_geo_draws):.1%} on average over twelve draws "
      f"(always answering green: {(y == 'green').mean():.1%})")

**Result.** 62% in this draw, 56% on average, against 51% for always answering `green`. Neighbours are alike, but 30 labelled hexagons cannot know where the forest ends.

**Next:** [self-supervised learning](self_supervised)